In [3]:
import io, json, pickle, torch, hebbian_locomotion

CKPT = "/cs/student/project_msc/2025/rai/mdecastr/Isaac_Lab/isaac_lab_sandbox/workspace/hebbian_locomotion/checkpoints/GO1_NEW_WITH_4096_SPEED_1_NO_Z_sigma_2_NO_RANK_FITNESS_FOOT_CONTACT_HELTHY_BONUS_0\.1_HEB_07:04-21:32_499.pickle"  # <-- set

# Remap CUDA-saved storages to CPU so this opens on a GPU-less node too.
class _CPUUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "torch.storage" and name == "_load_from_bytes":
            return lambda b: torch.load(io.BytesIO(b), map_location="cpu")
        return super().find_class(module, name)

with open(CKPT, "rb") as f:
    data = _CPUUnpickler(f).load()

solver, models = data[0], data[1]
pop_mean_curve, best_sol_curve = data[2], data[3]
run_cfg = data[4] if len(data) > 4 else None      # 5th element = config capture

if run_cfg is not None:
    print(json.dumps(run_cfg, indent=2, default=str))
else:
    print("No run_cfg (pre-capture checkpoint) — introspecting objects:\n")
    es = {k: getattr(solver, k, None) for k in
          ("popsize", "learning_rate", "learning_rate_decay",
           "sigma_init", "sigma", "sigma_decay", "rank_fitness", "antithetic")}
    gl = getattr(models, "gamma_logit", None)
    net = {"class": type(models).__name__,
           "sizes": getattr(models, "architecture", None),
           "norm_mode": getattr(models, "norm_mode", None),
           "M": getattr(models, "M", None),
           "gamma_logit": float(gl.flatten()[0]) if gl is not None else None}
    print("ES :", json.dumps(es,  indent=2, default=str))
    print("Net:", json.dumps(net, indent=2, default=str))

# Training summary (independent of run_cfg)
n = int((pop_mean_curve != 0).sum())
if n:
    print(f"\nEpochs logged: {n} | final mean fitness: {pop_mean_curve[n-1]:.3f} "
          f"| final best: {best_sol_curve[n-1]:.3f}")

ModuleNotFoundError: No module named 'hebbian_locomotion'